# Análisis Exploratorio de Datos (EDA) - Lending Club Dataset

## 1. Cargar Datos y Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

# Configuración para visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
dataset_path = '/home/jules/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3/accepted_2007_to_2018Q4.csv.gz'

# Cargar el dataset. Usamos low_memory=False para evitar advertencias de dtype debido a inferencia mixta.
try:
    df = pd.read_csv(dataset_path, compression='gzip', low_memory=False)
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print(f"Error: El archivo no se encontró en la ruta: {dataset_path}")
except Exception as e:
    print(f"Ocurrió un error al cargar el dataset: {e}")

## 2. Inspección Inicial de los Datos

In [ ]:
if 'df' in locals():
    print("df.head():")
    display(df.head())
    print("\n")
    
    print("df.info():")
    df.info()
    print("\n")
    
    print("df.shape:")
    print(df.shape)
    print("\n")
    
    print("df.describe(include='all'):")
    display(df.describe(include='all'))

## 3. Análisis de la Variable Objetivo (`loan_status`)

In [ ]:
if 'df' in locals():
    print("Distribución original de 'loan_status':")
    print(df['loan_status'].value_counts(normalize=True) * 100)
    print("\n")
    
    # Definir los estados que consideramos como 'default'
    default_statuses = ['Charged Off', 'Default', 
                        'Does not meet the credit policy. Status:Charged Off']
    
    # Crear la variable objetivo binaria 'is_default'
    df['is_default'] = df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)
    
    print("Distribución de la nueva variable objetivo 'is_default':")
    print(df['is_default'].value_counts(normalize=True) * 100)
    
    # Visualización de la distribución de 'loan_status'
    plt.figure(figsize=(10, 6))
    sns.countplot(y='loan_status', data=df, order = df['loan_status'].value_counts().index)
    plt.title('Distribución de Estatus de Préstamos (loan_status)')
    plt.xlabel('Cantidad')
    plt.ylabel('Estatus del Préstamo')
    plt.show()
    
    # Visualización de la distribución de 'is_default'
    plt.figure(figsize=(6, 4))
    sns.countplot(x='is_default', data=df)
    plt.title('Distribución de la Variable Objetivo (is_default)')
    plt.xlabel('Es Default (1 = Sí, 0 = No)')
    plt.ylabel('Cantidad')
    plt.xticks([0, 1], ['No Default (0)', 'Default (1)'])
    plt.show()

### Comentarios sobre la Variable Objetivo:
La variable original `loan_status` tiene múltiples categorías. Para nuestro modelo de predicción de incumplimiento, hemos creado una variable binaria `is_default`.
- **0 (No Default)**: Incluye préstamos que están 'Fully Paid', 'Current', 'In Grace Period', 'Late (16-30 days)', 'Late (31-120 days)' (estos últimos podrían ser considerados para un análisis más granular de riesgo, pero para un 'default' estricto, se excluyen por ahora) y 'Does not meet the credit policy. Status:Fully Paid'.
- **1 (Default)**: Incluye préstamos que están 'Charged Off', 'Default', y 'Does not meet the credit policy. Status:Charged Off'.

La distribución muestra un desbalance considerable, con un porcentaje mucho mayor de préstamos que no están en default. Esto es común en datasets de crédito y deberá ser considerado durante el modelado (e.g., usando técnicas de muestreo o métricas de evaluación apropiadas).

## 4. Limpieza Preliminar de Datos

In [ ]:
if 'df' in locals():
    # Identificar columnas con alto porcentaje de valores faltantes
    missing_values = df.isnull().mean() * 100
    high_missing_cols = missing_values[missing_values > 50]
    print("Columnas con más del 50% de valores faltantes:")
    print(high_missing_cols.sort_values(ascending=False))
    print(f"\nTotal de columnas con >50% de valores faltantes: {len(high_missing_cols)}\n")
    
    # Identificar columnas con un solo valor único (más la NaN si existe)
    single_unique_cols = []
    for col in df.columns:
        # nunique() no cuenta NaN por defecto. Si solo hay un valor y NaN, es un solo valor útil.
        # Si la columna está llena de NaNs, nunique() es 0. Si tiene un valor y NaNs, es 1. Si solo tiene un valor, es 1.
        if df[col].nunique(dropna=False) <= 2 and df[col].isnull().all(): # Toda la columna es NaN
             single_unique_cols.append(col)
        elif df[col].nunique(dropna=True) == 1: # Solo un valor único no-NaN
            single_unique_cols.append(col)
        elif df[col].nunique(dropna=True) == 0 and not df[col].isnull().all(): # Caso raro: no hay valores no-NaN pero no todos son NaN (imposible si hay datos)
             single_unique_cols.append(col)

    print("\nColumnas con un solo valor único (o todas NaN):")
    if single_unique_cols:
        for col in single_unique_cols:
            print(f"- {col}: {df[col].unique()}")
        print(f"\nTotal de columnas con un solo valor único: {len(single_unique_cols)}")
    else:
        print("No se encontraron columnas con un solo valor único.")

### Comentarios sobre Limpieza Preliminar:
Se identificaron varias columnas con un alto porcentaje de valores faltantes (más del 50%). Estas columnas son candidatas a ser eliminadas ya que imputar tantos valores podría introducir ruido. Columnas como `member_id`, `desc`, `sec_app_`, `hardship_` y varias relacionadas con información de crédito muy específica (`mths_since_last_major_derog`, `revol_bal_joint`, etc.) caen en esta categoría.

Adicionalmente, se buscaron columnas con un solo valor único. Columnas como `policy_code` (siempre es 1.0 para los datos aceptados que estamos usando) o `pymnt_plan` (casi siempre 'n') podrían no aportar información variada para el modelo. Estas también serían candidatas a eliminación. La inspección detallada del `df.describe(include='all')` también ayuda a identificar estas columnas (e.g., columnas con `top` siendo el único valor y `freq` igual al total de filas no nulas).

## 5. Visualizaciones de Características

### 5.1 Características Numéricas Clave

In [ ]:
if 'df' in locals():
    numerical_features = ['loan_amnt', 'funded_amnt', 'int_rate', 'installment', 'annual_inc', 'dti']
    
    for feature in numerical_features:
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        sns.histplot(df[feature], kde=True, bins=50)
        plt.title(f'Histograma de {feature}')
        plt.xlabel(feature)
        plt.ylabel('Frecuencia')
        
        plt.subplot(1, 2, 2)
        sns.boxplot(y=df[feature])
        plt.title(f'Boxplot de {feature}')
        
        plt.tight_layout()
        plt.show()
        
        print(f"Comentarios sobre '{feature}':")
        # Comentarios específicos basados en la observación de cada gráfico
        if feature == 'loan_amnt' or feature == 'funded_amnt':
            print("- La distribución del monto del préstamo está sesgada a la derecha, con la mayoría de los préstamos concentrados en montos más bajos. Hay algunos picos en valores redondos (e.g., 10k, 20k, 30k).")
            print("- El boxplot muestra la presencia de muchos outliers en los montos más altos.")
        elif feature == 'int_rate':
            print("- La tasa de interés muestra una distribución multimodal, posiblemente reflejando diferentes grados de riesgo o tipos de productos. Se observan concentraciones alrededor del 7-8%, 10-12%, y 13-15%.")
            print("- El boxplot indica que hay tasas de interés bastante altas consideradas como outliers.")
        elif feature == 'installment':
            print("- El pago mensual (installment) está sesgado a la derecha, similar al monto del préstamo. La mayoría de los pagos son relativamente bajos.")
            print("- El boxplot confirma la presencia de outliers para pagos mensuales elevados.")
        elif feature == 'annual_inc':
            print("- El ingreso anual está fuertemente sesgado a la derecha. La gran mayoría de los prestatarios tienen ingresos más bajos, con unos pocos teniendo ingresos muy altos.")
            print("- El boxplot muestra una gran cantidad de outliers en el extremo superior, lo que es típico para variables de ingresos. Considerar una transformación logarítmica para visualización o modelado podría ser útil.")
        elif feature == 'dti':
            print("- La relación deuda-ingreso (DTI) parece tener una distribución más o menos simétrica, centrada alrededor de 15-25. Hay una cola larga hacia valores más altos.")
            print("- El boxplot muestra outliers en ambos extremos, pero especialmente en el superior. Valores de DTI muy altos (o incluso negativos, si presentes y válidos) requerirán investigación.")
        print("\n")

### 5.2 Características Categóricas Clave

In [ ]:
if 'df' in locals():
    categorical_features = ['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 
                            'verification_status', 'purpose']
    
    for feature in categorical_features:
        plt.figure(figsize=(10, 6))
        # Para 'sub_grade' y 'purpose' que tienen muchas categorías, es mejor un gráfico de barras horizontal
        if feature in ['sub_grade', 'purpose']:
            sns.countplot(y=feature, data=df, order=df[feature].value_counts().index)
            plt.title(f'Distribución de {feature}')
            plt.xlabel('Cantidad')
            plt.ylabel(feature)
        else:
            sns.countplot(x=feature, data=df, order=df[feature].value_counts().index)
            plt.title(f'Distribución de {feature}')
            plt.xlabel(feature)
            plt.ylabel('Cantidad')
            if feature == 'emp_length': # Rotar etiquetas para mejor visualización
                 plt.xticks(rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
        
        print(f"Comentarios sobre '{feature}':")
        if feature == 'term':
            print("- La mayoría de los préstamos son a 36 meses, aunque hay una cantidad considerable a 60 meses.")
        elif feature == 'grade' or feature == 'sub_grade':
            print("- Los grados (grade) más comunes son B y C. Los subgrados (sub_grade) muestran una distribución más granular, con picos en B3, B4, C1, C2, etc.")
            print("- Esto indica una concentración de préstamos en categorías de riesgo moderado.")
        elif feature == 'emp_length':
            print("- La categoría más frecuente para la antigüedad laboral es '10+ years', seguida por '< 1 year' y '2 years'.")
            print("- La categoría '< 1 year' podría indicar un riesgo mayor, mientras que '10+ years' podría asociarse con mayor estabilidad.")
        elif feature == 'home_ownership':
            print("- Las categorías predominantes son 'MORTGAGE' (hipoteca) y 'RENT' (alquiler). 'OWN' (propia) es menos común.")
        elif feature == 'verification_status':
            print("- El estado de verificación más común es 'Source Verified', seguido de 'Not Verified' y 'Verified'.")
        elif feature == 'purpose':
            print("- El propósito principal para los préstamos es 'debt_consolidation' (consolidación de deudas), seguido por 'credit_card' (pago de tarjeta de crédito). Otros propósitos como 'home_improvement', 'other', y 'major_purchase' son menos frecuentes.")
        print("\n")

## 6. Análisis de Valores Faltantes

In [ ]:
if 'df' in locals():
    missing_percentage = (df.isnull().sum() / len(df)) * 100
    missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(x=missing_percentage.index, y=missing_percentage.values)
    plt.xticks(rotation=90)
    plt.title('Porcentaje de Valores Faltantes por Columna (solo columnas con faltantes)')
    plt.xlabel('Columnas')
    plt.ylabel('Porcentaje Faltante (%)')
    plt.tight_layout()
    plt.show()
    
    print("\nColumnas con valores faltantes y su porcentaje:")
    print(missing_percentage)

### Discusión sobre Estrategias de Imputación (Preliminar):
Para las columnas que se decida conservar y que tengan valores faltantes:
- **Características Numéricas**: 
  - Si la distribución es aproximadamente simétrica, se podría usar la **media**.
  - Si la distribución está sesgada o tiene muchos outliers (e.g., `annual_inc`, `mths_since_last_delinq`), la **mediana** sería una opción más robusta.
  - Para algunas variables, un valor específico como 0 podría tener sentido (e.g., `mths_since_last_delinq` podría indicar que nunca hubo delincuencia, aunque esto requiere un análisis más profundo del significado de NaN en cada caso).
- **Características Categóricas**:
  - La **moda** (el valor más frecuente) es una estrategia común.
  - Crear una categoría separada como 'Desconocido' o 'Faltante' también es una opción, especialmente si el hecho de que falte el valor es informativo.

La elección final de la estrategia de imputación dependerá del análisis detallado de cada variable y su relación con la variable objetivo, lo cual se abordará en la etapa de ingeniería de características.

## 7. Análisis de Correlación (Características Numéricas)

In [ ]:
if 'df' in locals():
    # Seleccionar solo columnas numéricas para la matriz de correlación
    # Excluir 'id' y 'member_id' si existen y son solo identificadores, y la variable objetivo 'is_default'
    cols_to_exclude = ['id', 'member_id', 'is_default'] 
    numerical_cols_for_corr = df.select_dtypes(include=np.number).columns.tolist()
    numerical_cols_for_corr = [col for col in numerical_cols_for_corr if col not in cols_to_exclude]
    
    correlation_matrix = df[numerical_cols_for_corr].corr()
    
    plt.figure(figsize=(20, 18))
    sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Heatmap de Correlación de Características Numéricas')
    plt.show()
    
    print("\nComentarios sobre Correlaciones Altas:")
    print("- `loan_amnt`, `funded_amnt`, y `funded_amnt_inv` están altamente correlacionadas (cercano a 1). Esto es esperado, ya que representan el monto solicitado, el monto fondeado por LendingClub, y el monto fondeado por inversores. Se podría conservar solo `loan_amnt` o la que se considere más relevante.")
    print("- `installment` está altamente correlacionada con `loan_amnt` y `funded_amnt`. El pago mensual depende directamente del monto del préstamo.")
    print("- `fico_range_low` y `fico_range_high` están perfectamente correlacionadas (1.00), lo cual es lógico. Solo una es necesaria. Similarmente, `last_fico_range_high` y `last_fico_range_low`.")
    print("- `total_pymnt` y `total_rec_prncp` suelen estar altamente correlacionadas, ya que el pago total incluye el capital recuperado.")
    print("- Algunas variables relacionadas con el balance total de crédito (`tot_hi_cred_lim`, `total_bal_ex_mort`) pueden mostrar correlaciones con otras variables de saldo o límites.")
    print("Estas altas correlaciones sugieren posible multicolinealidad, lo que podría ser un problema para algunos modelos (e.g., regresión lineal). La selección de características o técnicas como PCA podrían ser útiles.")

## 8. Resumen de Observaciones y Tendencias del EDA (en Español Mexicano)

1.  **Calidad de los Datos y Variable Objetivo**:
    *   El dataset es bastante grande, con más de 2.2 millones de registros y 151 columnas.
    *   Se creó una variable objetivo binaria `is_default` (1 para 'Charged Off' o 'Default', 0 para el resto). La proporción de 'default' es de aproximadamente 11.7%, indicando un desbalance de clases que hay que manejar en el modelado.
    *   Existen numerosas columnas con un alto porcentaje de valores faltantes (más del 50%), especialmente aquellas relacionadas con aplicaciones conjuntas (`sec_app_`, `revol_bal_joint`, `dti_joint`, etc.), programas de dificultad (`hardship_`) y algunas métricas de crédito específicas (`desc`, `member_id`, `mths_since_last_major_derog`). Estas columnas probablemente serán eliminadas.
    *   Se identificaron columnas con un solo valor único (e.g., `policy_code`, `pymnt_plan` en su mayoría) que no aportarán varianza al modelo y pueden ser eliminadas.

2.  **Distribuciones de Características Clave**:
    *   **Monto del Préstamo (`loan_amnt`)**: La mayoría de los préstamos son de montos menores (entre 5,000 y 20,000 USD), con picos en cifras redondas. La distribución está sesgada a la derecha.
    *   **Tasa de Interés (`int_rate`)**: Presenta una distribución multimodal, sugiriendo diferentes perfiles de riesgo. Las tasas más comunes se agrupan en ciertos rangos (e.g., 7-9%, 10-13%, 14-16%).
    *   **Ingreso Anual (`annual_inc`)**: Fuertemente sesgada a la derecha. La mayoría de los solicitantes tienen ingresos moderados, pero hay valores atípicos muy altos. Se podría necesitar una transformación logarítmica para el modelado.
    *   **Plazo del Préstamo (`term`)**: Predominan los préstamos a 36 meses sobre los de 60 meses.
    *   **Grado del Préstamo (`grade`, `sub_grade`)**: Los grados B y C son los más comunes, indicando una concentración en riesgo crediticio moderado.
    *   **Propósito del Préstamo (`purpose`)**: La consolidación de deudas (`debt_consolidation`) y el pago de tarjetas de crédito (`credit_card`) son los motivos más frecuentes.
    *   **Antigüedad Laboral (`emp_length`)**: La categoría más común es '10+ years', lo que podría indicar estabilidad laboral para una buena parte de los solicitantes.

3.  **Valores Atípicos (Outliers)**:
    *   Varias características numéricas importantes como `loan_amnt`, `annual_inc`, y `installment` muestran la presencia de valores atípicos significativos, especialmente en el extremo superior. Su tratamiento (e.g., clipping, transformación) será importante.

4.  **Correlaciones**:
    *   Se observan altas correlaciones entre variables que son inherentemente similares (e.g., `loan_amnt`, `funded_amnt`, `funded_amnt_inv`; `fico_range_low`, `fico_range_high`). Esto es natural y se deberá gestionar para evitar multicolinealidad, por ejemplo, seleccionando una de las variables del grupo correlacionado.
    *   `installment` está fuertemente correlacionada con `loan_amnt`, lo cual es lógico.

5.  **Próximos Pasos Sugeridos (Hacia Ingeniería de Características)**:
    *   Eliminar columnas con demasiados valores faltantes o con varianza nula/casi nula.
    *   Imputar valores faltantes en las columnas restantes usando estrategias adecuadas (mediana/media para numéricas, moda/categoría especial para categóricas).
    *   Realizar transformaciones de datos (e.g., logarítmica para `annual_inc`) para normalizar distribuciones o reducir el impacto de outliers.
    *   Convertir características categóricas a formato numérico (e.g., one-hot encoding, label encoding).
    *   Considerar la creación de nuevas características a partir de las existentes (e.g., ratios, interacciones).
    *   Manejar el desbalance de la variable objetivo (`is_default`) mediante técnicas de muestreo (oversampling, undersampling, SMOTE) o usando métricas de evaluación robustas al desbalance (e.g., F1-score, AUC-PR).